In [1]:
#!/usr/bin/env python3
"""
ingest_keymetrics_ttm_secuencial_auditado.py

Ingesta SECUENCIAL del endpoint key-metrics-ttm (FMP)
+ auditoría completa en infraestructura.update_logs
"""

import os
import time
import requests
import logging
import subprocess
from datetime import date
from dotenv import load_dotenv
import psycopg2

# ---------------------------
# ENV
# ---------------------------
load_dotenv()

POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_HOST     = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT", "5432")
API_KEY           = os.getenv("FMP_API_KEY")

if not all([POSTGRES_DB, POSTGRES_USER, POSTGRES_PASSWORD, API_KEY]):
    raise EnvironmentError("Faltan variables de entorno")

# ---------------------------
# Logging
# ---------------------------
LOG_DIR  = "output_ingest"
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = f"{LOG_DIR}/ingest_keymetrics_ttm_{date.today().isoformat()}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)

logging.info("=== START ingest_keymetrics_ttm_secuencial ===")

# ---------------------------
# Keep Mac awake
# ---------------------------
try:
    caffeinate = subprocess.Popen(["caffeinate"])
except Exception:
    caffeinate = None

# ---------------------------
# DB helpers
# ---------------------------
def get_conn():
    return psycopg2.connect(
        dbname=POSTGRES_DB,
        user=POSTGRES_USER,
        password=POSTGRES_PASSWORD,
        host=POSTGRES_HOST,
        port=POSTGRES_PORT
    )

def log_db(conn, ticker, status, message):
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO infraestructura.update_logs
            (schema_name, table_name, ticker, status, message)
            VALUES (%s, %s, %s, %s, %s)
        """, (
            'api_raw',
            'multifactor_keymetrics_ttm',
            ticker,
            status,
            message
        ))
        conn.commit()

# ---------------------------
# Fetch tickers
# CAMBIO 1: ahora desde universos.stock_opciones_2026
# ---------------------------
def obtener_tickers(conn):
    with conn.cursor() as cur:
        cur.execute("SELECT ticker FROM universos.stock_opciones_2026")
        return [r[0] for r in cur.fetchall()]

# ---------------------------
# API
# CAMBIO 2: nuevo endpoint /stable/ratios-ttm
# ---------------------------
def fetch_keymetrics_ttm(ticker):
    url = f"https://financialmodelingprep.com/stable/key-metrics-ttm?symbol={ticker}&apikey={API_KEY}"
    r = requests.get(url, timeout=20)

    if r.status_code != 200:
        return None, f"HTTP_{r.status_code}"

    data = r.json()
    if not data:
        return None, "NO_DATA"

    return data[0], "OK"

# ---------------------------
# Insert
# CAMBIO 3: nombres de métricas actualizados
#   marketCapTTM              → marketCap
#   roicTTM                   → returnOnInvestedCapitalTTM
# El INSERT guarda en las columnas originales de la tabla (no se renombran)
# pero el payload mapea los nuevos nombres que devuelve la API
# ---------------------------
INSERT_SQL = """
INSERT INTO api_raw.multifactor_keymetrics_ttm (
    ticker,
    fecha_de_consulta,
    freeCashFlowYieldTTM,
    marketCapTTM,
    netDebtToEBITDATTM,
    roicTTM
)
VALUES (
    %(ticker)s,
    %(fecha)s,
    %(freeCashFlowYieldTTM)s,
    %(marketCap)s,
    %(netDebtToEBITDATTM)s,
    %(returnOnInvestedCapitalTTM)s
)
ON CONFLICT (ticker, fecha_de_consulta)
DO UPDATE SET
    freeCashFlowYieldTTM = EXCLUDED.freeCashFlowYieldTTM,
    marketCapTTM         = EXCLUDED.marketCapTTM,
    netDebtToEBITDATTM   = EXCLUDED.netDebtToEBITDATTM,
    roicTTM              = EXCLUDED.roicTTM,
    updated_at           = CURRENT_TIMESTAMP;
"""

# ---------------------------
# MAIN
# ---------------------------
def main():
    conn    = get_conn()
    tickers = obtener_tickers(conn)
    total   = len(tickers)

    logging.info(f"Tickers a procesar: {total}")

    ok, fail = 0, 0

    for i, ticker in enumerate(tickers, 1):
        try:
            data, status = fetch_keymetrics_ttm(ticker)

            if not data:
                log_db(conn, ticker, status, "Fetch failed")
                fail += 1
                continue

            # Mapeo explícito: nombre API → nombre columna en payload
            payload = {
                "ticker"                     : ticker,
                "fecha"                      : date.today(),
                "freeCashFlowYieldTTM"       : data.get("freeCashFlowYieldTTM"),
                "marketCap"                  : data.get("marketCap"),
                "netDebtToEBITDATTM"         : data.get("netDebtToEBITDATTM"),
                "returnOnInvestedCapitalTTM" : data.get("returnOnInvestedCapitalTTM"),
            }

            with conn.cursor() as cur:
                cur.execute(INSERT_SQL, payload)
                conn.commit()

            log_db(conn, ticker, "success", "Inserted keymetrics_ttm")
            ok += 1

        except Exception as e:
            conn.rollback()
            log_db(conn, ticker, "exception", str(e))
            fail += 1
            logging.warning(f"{ticker} error: {e}")

        if i % 100 == 0:
            logging.info(f"Procesados {i}/{total}")

        time.sleep(0.25)

    conn.close()
    logging.info(f"FIN | OK={ok} | FAIL={fail}")

# ---------------------------
if __name__ == "__main__":
    try:
        main()
    finally:
        if caffeinate:
            caffeinate.terminate()

2026-04-07 09:57:11,701 | INFO | === START ingest_keymetrics_ttm_secuencial ===
2026-04-07 09:57:11,847 | INFO | Tickers a procesar: 3027
2026-04-07 09:58:49,288 | INFO | Procesados 100/3027
2026-04-07 10:00:28,992 | INFO | Procesados 200/3027
2026-04-07 10:02:07,042 | INFO | Procesados 300/3027
2026-04-07 10:03:44,522 | INFO | Procesados 400/3027
2026-04-07 10:05:22,117 | INFO | Procesados 500/3027
2026-04-07 10:07:00,194 | INFO | Procesados 600/3027
2026-04-07 10:08:38,239 | INFO | Procesados 700/3027
2026-04-07 10:10:15,976 | INFO | Procesados 800/3027
2026-04-07 10:11:52,916 | INFO | Procesados 900/3027
2026-04-07 10:13:30,393 | INFO | Procesados 1000/3027
2026-04-07 10:15:08,330 | INFO | Procesados 1100/3027
2026-04-07 10:16:46,251 | INFO | Procesados 1200/3027
2026-04-07 10:18:23,946 | INFO | Procesados 1300/3027
2026-04-07 10:20:01,796 | INFO | Procesados 1400/3027
2026-04-07 10:21:42,138 | INFO | Procesados 1500/3027
2026-04-07 10:23:18,675 | INFO | Procesados 1600/3027
2026-04